# One paper, two readings

### Is "functional evidence" a stable category in ClinVar?

A sentence-level audit of how laboratories describe functional evidence across 13 RASopathy genes,
followed by a robustness check of the pipeline that produced the counts.

**The question.** Two laboratories can cite the same paper on the same variant and record opposite
conclusions about whether functional evidence exists. This notebook measures how often that happens,
reads the cases it finds one by one, and then asks whether its own measurement survives the defects
that two independent code reviews found in it.

| § | Establishes |
|---|---|
| 1 | Whether a bibliography-controlled comparison is possible at all — and the clustering behind any rate |
| 2 | Whether claims and denials cite literature at the same rate, inside laboratories as well as across them |
| 3 | The worked example: which papers each submitter on SOS1 p.Ile437Thr names |
| 4 | Which sentence each citation sits in, reconciled against the published classifier |
| 5 | Reading the flagged cases — four found mechanically, two survive |
| 6 | Four pipeline defects measured against the published counts |
| 7 | The robustness result, as figures |

**The outputs below are stored** from the run of 26–29 September 2026 — 6,738,642 rows scanned,
32,562 panel submissions, 17,369 variants, 13 genes. Four independent reproductions on the same
ClinVar release reproduce every reported number. Nothing has to be re-run to read this notebook.

**To re-run it**, cells execute in order in one kernel, and three things must be in place:

- **Internet enabled.** Section 1 bootstraps the classifier notebook and downloads ClinVar's
  `submission_summary.txt.gz` from the NCBI FTP site — **395 MB**. The bootstrapped cells' own
  output is held in `_bootstrap_log` rather than printed; `print(_bootstrap_log)` to read it.
- **The classifier notebook attached as an input**, so the bootstrap can find it:
  *Add Input → Notebooks →* `are-functional-evidence-claims-in-clinvar-auditabl`. Alternatively,
  paste these cells at the end of that notebook, where `subs` and `classify` already exist and no
  bootstrap runs.
- **Optionally, the three labelled CSVs.** Without them the classifier's accuracy validation is
  skipped — it prints `NOT FOUND` and continues. Nothing in this notebook depends on it.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/__results__.html
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/submission_summary.txt.gz
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/__notebook__.ipynb
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/__output__.json
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/custom.css
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/panel_claims_v2/classified.csv
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/panel_claims_v2/by_gene.csv
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/panel_claims_v2/by_submitter.csv
/kaggle/input/notebooks/fernandosr85/are-functional-evidence-claims-in-clinvar-auditabl/panel_claims_v2/summary.jso

## 1 · Is a bibliography-controlled comparison even possible?

Before asking how often submitters disagree about evidence they share, check whether they ever
*do* share it. This cell pairs submissions on the same variant whose citation sets overlap, then
reports the pair counts.

**Section 3 of the output is the one that decides everything below it.** A submitter's wording is a
house template, not a per-variant judgement: if two laboratories coincide on hundreds of variants,
every one contributes a pair, and a mean over pairs reports two templates replicated, dressed as a
population rate. The count of *distinct submitter pairs* is the honest denominator.

The PMID parser opens with a structural test. Submitters write citation lists two ways — anchoring
every PMID, or anchoring once and comma-separating the rest — and a pattern that handles only one of
them silently undercounts, which would make sets look disjoint and manufacture the result the
hypothesis predicts.


In [2]:
"""
Feasibility diagnostic for an Interpretation-Divergence-under-Fixed-Evidence
study, over the 13-gene RASopathy panel.

Run this BEFORE computing any divergence metric. It answers one question:
is "two submitters citing the same evidence" common enough, and independent
enough, to support a rate — or is it a case series?

WHERE THIS BELONGS
    Appendix cell for the classifier notebook (panel_functional_claims_v6).
    It needs `subs` and `classify`, built by that notebook's first four code
    cells. If the kernel is empty it bootstraps itself by executing those
    cells from a copy of that notebook found on disk — on Kaggle, attach it
    with Add Input → Notebooks. Set BOOTSTRAP = False to forbid that.

    Bootstrapping re-executes the real cells rather than reimplementing them.
    A second copy of the classifier would drift from the validated one and
    the numbers would stop matching the study.

    The bootstrap must never execute THIS cell; two guards below make that
    impossible. See the comment on _CELL_SIGNATURE.

WHAT IT REUSES, AND WHY IT DOES NOT RECOMPUTE
    After the fourth code cell, `subs` already carries `text` (Description +
    ExplanationOfInterpretation, placeholder-stripped) and the boolean flags
    `claim` / `denial`. This cell reads those columns instead of re-deriving
    them, so the label basis is identical to the published figures.
    Rebuilding the text here would silently classify a different string.

    Note the contract: the classifier returns {"claim", "denial",
    "prediction", "lit_absence"} — booleans, not a label. The four strata
    below are derived from the claim/denial pair, matching POP_STRATA in the
    notebook.

OUTPUT, in order of what it decides
    1. coverage     — how many submissions cite anything at all
    2. pairs        — variants where two submitters cite an equivalent set,
                      counted at BOTH the variant level and the submitter-pair
                      level, because the clustering is what decides whether a
                      panel-wide rate means anything
    3. divergence   — reported at both levels, so the inflation is visible
    4. propagation  — independently curated bibliographies, or inherited ones?
    5. sensitivity  — the headline must not rest on one arbitrary threshold

    Printing is deliberately terse. The full pair-level data goes to
    bibliography_pairs.csv and divergence_by_submitter_pair.csv, and the
    bootstrapped cells' own output is kept in `_bootstrap_log`.
"""

import contextlib
import glob
import io
import itertools
import json
import os
import re

import pandas as pd

# --------------------------------------------------------------------------
# Parameters. MIN_CITES is the one that matters: two submissions each citing a
# single identical PMID score Jaccard 1.0, a far weaker control than the 11-of-11
# seen on SOS1 VCV000045345. Section 5 sweeps both; do not report one cell of it.
# --------------------------------------------------------------------------
MIN_CITES = 3
JACCARD_EQUIV = 0.80

BOOTSTRAP = True
BOOTSTRAP_CELLS = 4          # cells 0-3: constants, regexes, classifier, load

# --------------------------------------------------------------------------
# Finding the classifier notebook.
#
# Match on content, not filename: the notebook is saved locally as
# panel_functional_claims_v6.ipynb and on Kaggle as the slug of its title, so
# a filename match picks the wrong copy or none. The markers below appear
# together only in that classifier. Verified across six downloaded Kaggle
# copies: code cells 0-2 are byte-identical in all of them, and cell 3 differs
# only in the stratum-weighting fix, so any copy bootstraps to the same
# contract.
#
# The marker strings are ASSEMBLED here and never appear as literals in this
# cell. Written out in full, they made this cell match its own marker list:
# the newest "classifier" on disk became the notebook containing this cell,
# and the bootstrap exec'd the bootstrapper. Kaggle hits that the moment Save
# & Run All writes /kaggle/working/__notebook__.ipynb — RecursionError, not at
# run time but at save time. The description of a thing was read as the thing;
# same failure as reading a domain annotation as the molecule.
# --------------------------------------------------------------------------
BOOTSTRAP_MARKERS = ("def " + "load_panel", "def " + "classify(",
                     "LITERATURE" + "_ABSENCE", "VENDOR" + "_MODEL")

# Second guard, independent of the first: any code cell whose source contains
# this identifier belongs to this appendix and is never executed by the
# bootstrap. The identifier is its own signature, so it survives editing and
# keeps the documented "paste this cell at the end of the classifier notebook"
# arrangement safe.
_CELL_SIGNATURE = "_CELL" + "_SIGNATURE"

# Third guard: a re-entry flag. It lives in os.environ rather than in globals()
# because the bootstrap exec's into globals(), so a global sentinel could be
# reset by the very cell it is meant to stop.
_BOOTSTRAP_FLAG = "CLINVAR_BOOTSTRAP_ACTIVE"

# --------------------------------------------------------------------------
# Submitters write citation lists two ways, and a pattern that handles only one
# of them silently undercounts — which would make sets look disjoint and inflate
# apparent divergence, i.e. manufacture the result the hypothesis predicts:
#   Dasa    "(PMID: 21387466; PMID: 24803665; PMID: 30541462)"   anchor each
#   GeneDx  "(PMID: 24803665, 30541462, 30039904, 24451042)"     anchor once
# A naive r"PMID:?\s*(\d{7,8})" returns 3 and 1. Anchor on PMID, then consume the
# whole delimited run. The structural test below fails if this regresses.
# --------------------------------------------------------------------------
PMID_BLOCK = re.compile(
    r"PMIDs?\s*:?\s*(\d{7,9}(?:\s*(?:,|;|and|&|/)\s*(?:PMIDs?\s*:?\s*)?\d{7,9})*)",
    re.I)
PMID_NUM = re.compile(r"\d{7,9}")


def cited_pmids(text):
    out = set()
    for m in PMID_BLOCK.finditer(text or ""):
        out.update(PMID_NUM.findall(m.group(1)))
    return frozenset(out)


def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def _test_pmid_parsing():
    """Both list styles must yield the same set; a year is never a PMID.

    Silent on success — the assertions are the report.
    """
    cases = [
        ("(PMID: 24803665, 30541462, 30039904, 24451042, 21387466)",
         {"24803665", "30541462", "30039904", "24451042", "21387466"}),
        ("(PMID: 21387466; PMID: 24803665; PMID: 30541462; PMID: 30039904; "
         "PMID: 24451042).",
         {"21387466", "24803665", "30541462", "30039904", "24451042"}),
        ("reported in 2018 and 2021 by several groups", set()),
        ("PMIDs 12628188 and 17143282", {"12628188", "17143282"}),
        ("no citation here", set()),
    ]
    for text, want in cases:
        got = set(cited_pmids(text))
        assert got == want, f"PMID parsing broke on {text!r}: {got} != {want}"
    assert cases[0][1] == cases[1][1], "the two styles must agree"


_test_pmid_parsing()


# --------------------------------------------------------------------------
# Bootstrap + preflight.
# --------------------------------------------------------------------------
def _notebook_code(path):
    """Code cells of a notebook, with this appendix's own cells removed.

    Returns None if the file is not a notebook. `source` is a string in some
    notebooks and a list of lines in others; both are normalised here.
    """
    try:
        with open(path, encoding="utf-8") as fh:
            nb = json.load(fh)
    except Exception:
        return None
    cells = []
    for c in nb.get("cells", []):
        if c.get("cell_type") != "code":
            continue
        src = c.get("source", "")
        if not isinstance(src, str):
            src = "".join(src)
        if _CELL_SIGNATURE in src:
            continue                     # this appendix — never bootstrap it
        cells.append(src)
    return cells


def _find_classifier_notebook():
    """Newest .ipynb whose remaining code carries every marker."""
    seen, found = set(), []
    for pattern in ("/kaggle/input/**/*.ipynb", "/kaggle/working/**/*.ipynb",
                    "./**/*.ipynb"):
        for p in glob.glob(pattern, recursive=True):
            rp = os.path.realpath(p)
            if rp in seen or not os.path.isfile(rp):
                continue
            seen.add(rp)
            code = _notebook_code(rp)
            if not code:
                continue
            src = "\n".join(code)
            if all(m in src for m in BOOTSTRAP_MARKERS):
                found.append((os.path.getmtime(rp), rp, code))
    if not found:
        return None, None
    found.sort(reverse=True)
    return found[0][1], found[0][2]


def _label(path):
    """Kaggle names every notebook __notebook__.ipynb; show its directory."""
    return os.path.join(os.path.basename(os.path.dirname(path)),
                        os.path.basename(path))


def _bootstrap():
    if os.environ.get(_BOOTSTRAP_FLAG):
        raise RuntimeError(
            "bootstrap re-entered — the notebook selected for bootstrapping "
            "contains this appendix cell, which should have been filtered by "
            "_CELL_SIGNATURE. Check that both guards above are intact.")
    path, code = _find_classifier_notebook()
    if path is None:
        raise RuntimeError(
            "no notebook carrying the classifier was found under "
            "/kaggle/input, /kaggle/working or the working directory. On "
            "Kaggle the running notebook is not on disk, so attach the "
            "classifier notebook as an input (Add Input → Notebooks), or "
            "paste this cell at the END of that notebook instead — then "
            "`subs` and the classifier already exist and no bootstrap is "
            "needed.")
    os.environ[_BOOTSTRAP_FLAG] = "1"
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            for i, src in enumerate(code[:BOOTSTRAP_CELLS]):
                exec(compile(src, f"{_label(path)}[code cell {i}]", "exec"),
                     globals())
    except Exception:
        print(buf.getvalue()[-2000:])    # the suppressed output explains it
        raise
    finally:
        os.environ.pop(_BOOTSTRAP_FLAG, None)
        globals()["_bootstrap_log"] = buf.getvalue()
    print(f"bootstrap: code cells 0-{BOOTSTRAP_CELLS - 1} of {_label(path)} "
          f"({len(buf.getvalue().splitlines())} lines of their output held in "
          "_bootstrap_log)")


def _preflight():
    """Silent on success. Every check below failed at least once in drafting."""
    if "subs" not in globals() or "classify" not in globals():
        missing = [n for n in ("subs", "classify") if n not in globals()]
        if not BOOTSTRAP:
            raise RuntimeError(
                f"not defined in this kernel: {', '.join(missing)}. Run the "
                "first four code cells of the classifier notebook, or set "
                "BOOTSTRAP = True.")
        _bootstrap()

    df = globals()["subs"]
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"`subs` is a {type(df).__name__}, expected a DataFrame")

    required = {"VariationID", "Submitter", "gene", "DateLastEvaluated"}
    absent = required - set(df.columns)
    if absent:
        raise KeyError(f"`subs` is missing {sorted(absent)}. "
                       f"Present: {list(df.columns)}")

    # The classifier returns booleans, not a label. Assert the real contract:
    # an earlier draft of this cell assumed a "label" key and its own test
    # agreed with it, because the test's stub was written from the same
    # assumption.
    probe = globals()["classify"]("functional studies demonstrate a damaging effect")
    if not isinstance(probe, dict) or "claim" not in probe or "denial" not in probe:
        raise TypeError("the classifier must return a dict with 'claim' and "
                        f"'denial' keys; got {probe!r}")

    return {"text", "claim", "denial"} <= set(df.columns)


_have_flags = _preflight()


# --------------------------------------------------------------------------
# One row per submission: who, when, what they cited, what they asserted.
# --------------------------------------------------------------------------
work = subs.copy()

if not _have_flags:
    # Fallback only. Mirrors cell 3's basis exactly: Description joined with
    # ExplanationOfInterpretation, ClinVar's "-" placeholder treated as empty.
    ph = re.compile(r"^\s*[-–—.·]?\s*$")

    def _blank_local(s):
        s = s.fillna("").astype(str)
        return s.mask(s.str.match(ph), "")

    txt = _blank_local(work["Description"])
    if "ExplanationOfInterpretation" in work.columns:
        txt = txt + " " + _blank_local(work["ExplanationOfInterpretation"])
    work["text"] = txt.str.strip()
    flags = work["text"].apply(classify).apply(pd.Series)
    for c in ("claim", "denial"):
        work[c] = flags[c]


def _stratum(row):
    if row["claim"] and row["denial"]:
        return "both"
    if row["claim"]:
        return "claim"
    if row["denial"]:
        return "denial"
    return "neither"


work["label"] = work.apply(_stratum, axis=1)
work["pmids"] = work["text"].map(cited_pmids)
work["n_cites"] = work["pmids"].map(len)
work["date"] = pd.to_datetime(work["DateLastEvaluated"], errors="coerce",
                              format="mixed")


def _counts(vc):
    return "  ".join(f"{vc.get(k, 0):,} {k}"
                     for k in ("claim", "denial", "both", "neither"))


n_sub = len(work)
n_any = int((work.n_cites > 0).sum())
n_min = int((work.n_cites >= MIN_CITES).sum())
sizes = work.loc[work.n_cites > 0, "n_cites"]

print("1. COVERAGE")
print(f"  submissions in panel        : {n_sub:,}")
print(f"    labels                    : {_counts(work.label.value_counts())}")
print(f"  citing >= 1 PMID            : {n_any:,} ({100*n_any/n_sub:.1f}%)")
print(f"    labels                    : "
      f"{_counts(work.loc[work.n_cites > 0, 'label'].value_counts())}")
print(f"  citing >= {MIN_CITES} PMIDs           : {n_min:,} ({100*n_min/n_sub:.1f}%)")
print(f"  set size where non-zero     : median {sizes.median():.0f}, "
      f"90th pct {sizes.quantile(.9):.0f}, max {sizes.max():.0f}")

# --------------------------------------------------------------------------
# Pair submissions within each variant.
# --------------------------------------------------------------------------
elig = work[work.n_cites >= MIN_CITES]
rows = []
for vid, grp in elig.groupby("VariationID", sort=False):
    if len(grp) < 2:
        continue
    recs = grp.to_dict("records")
    for a, b in itertools.combinations(recs, 2):
        if a["Submitter"] == b["Submitter"]:
            continue
        j = jaccard(a["pmids"], b["pmids"])
        if j < JACCARD_EQUIV:
            continue
        first, second = sorted([a, b],
                               key=lambda r: (pd.isna(r["date"]), r["date"]))
        rows.append({
            "VariationID": vid, "gene": a["gene"],
            "sub_a": first["Submitter"], "sub_b": second["Submitter"],
            "label_a": first["label"], "label_b": second["label"],
            "n_a": len(first["pmids"]), "n_b": len(second["pmids"]),
            "jaccard": j,
            "identical": first["pmids"] == second["pmids"],
            "later_superset": first["pmids"] < second["pmids"],
            "date_a": first["date"], "date_b": second["date"],
            "gap_days": (second["date"] - first["date"]).days
                        if pd.notna(first["date"]) and pd.notna(second["date"])
                        else None,
        })

pairs = pd.DataFrame(rows)

print()
print(f"2. EQUIVALENT-BIBLIOGRAPHY PAIRS  "
      f"(both >= {MIN_CITES} cites, Jaccard >= {JACCARD_EQUIV:.2f})")
if pairs.empty:
    print("  none at these thresholds. IDFE is not computable on this panel;")
    print("  see the sweep in section 5 before concluding anything.")
else:
    # ----------------------------------------------------------------------
    # A submitter's wording is a house template, not a per-variant judgement.
    # If two labs coincide on hundreds of variants, every one contributes a
    # pair, and a mean over pairs reports two templates replicated, dressed as
    # a population rate. Same clustering that forced the per-laboratory split
    # in the original study — so the submitter-pair count below is the number
    # that decides whether any panel-wide rate means anything.
    # ----------------------------------------------------------------------
    pairs["lab_pair"] = [" | ".join(sorted([a, b]))
                         for a, b in zip(pairs.sub_a, pairs.sub_b)]
    n_lab_pairs = pairs.lab_pair.nunique()
    top = pairs.lab_pair.value_counts()

    print(f"  variant-level pairs         : {len(pairs):,}  "
          f"({int(pairs.identical.sum()):,} with identical citation sets)")
    print(f"  distinct variants           : {pairs.VariationID.nunique():,}")
    print(f"  distinct submitter pairs    : {n_lab_pairs:,}  -> "
          f"{len(pairs)/n_lab_pairs:.1f} variant-pairs each; largest "
          f"{100*top.iloc[0]/len(pairs):.0f}%, top 3 "
          f"{100*top.head(3).sum()/len(pairs):.0f}% of all pairs")

    disc = pairs[pairs.label_a != pairs.label_b]
    hard = pairs[((pairs.label_a == "claim") & (pairs.label_b == "denial")) |
                 ((pairs.label_a == "denial") & (pairs.label_b == "claim"))]
    by_lab = (pairs.assign(diff=pairs.label_a != pairs.label_b)
                   .groupby("lab_pair")["diff"].agg(["mean", "size"])
                   .sort_values("size", ascending=False))

    print()
    print("3. DIVERGENCE UNDER FIXED EVIDENCE")
    print(f"  variant-level, any difference   : {100*len(disc)/len(pairs):5.1f}%"
          "   <- inflated by clustering")
    print(f"  variant-level, claim vs denial  : {100*len(hard)/len(pairs):5.1f}%")
    print(f"  submitter-pair level, unweighted: {100*by_lab['mean'].mean():5.1f}%"
          f"   (n={n_lab_pairs} pairs)")
    print(f"  submitter pairs that ever differ: "
          f"{int((by_lab['mean'] > 0).sum()):,} / {n_lab_pairs:,}")
    print("  busiest submitter pairs:")
    for name, r in by_lab.head(3).iterrows():
        print(f"    {100*r['mean']:5.1f}% of {int(r['size']):>3}   {name}")

    # ----------------------------------------------------------------------
    # Identical bibliographies across independent labs would be a remarkable
    # coincidence; a shared upstream source (HGMD, a commercial annotation
    # pipeline, or ClinVar itself) is likelier. If the citations were inherited
    # then "same evidence, different label" is really "inherited citations,
    # different template" — a different claim, and not the one IDFE names.
    # ----------------------------------------------------------------------
    print()
    print("4. PROPAGATION CHECK")
    dated = pairs[pairs.gap_days.notna()]
    if len(dated):
        non_ident = dated[~dated.identical]
        sup = (f"{100*non_ident.later_superset.mean():.0f}%"
               if len(non_ident) else "n/a")
        med = (dated.assign(diff=dated.label_a != dated.label_b)
                    .groupby("diff")["gap_days"].median())
        print(f"  pairs with both dates       : {len(dated):,}")
        print(f"  identical sets              : {100*dated.identical.mean():.0f}%")
        print(f"  later a strict superset     : {sup}  "
              f"(of the {len(non_ident):,} non-identical)")
        print(f"  median gap                  : {dated.gap_days.median():.0f} days"
              f"   (agreeing {med.get(False, float('nan')):.0f}, "
              f"differing {med.get(True, float('nan')):.0f})")
    else:
        print("  no pairs with both dates parsed.")

    pairs.to_csv("bibliography_pairs.csv", index=False)
    by_lab.to_csv("divergence_by_submitter_pair.csv")
    print("  wrote bibliography_pairs.csv, divergence_by_submitter_pair.csv")

# --------------------------------------------------------------------------
# Sensitivity. Precomputed per threshold so the sweep is not quadratic in rows.
# --------------------------------------------------------------------------
print()
print("5. SENSITIVITY TO THRESHOLDS")
print(f"  {'min_cites':>9} {'jaccard':>8} {'var_pairs':>10} {'lab_pairs':>10} "
      f"{'differ%':>8}")
for mc in (1, 2, 3, 5):
    e = work[work.n_cites >= mc]
    cand = []
    for vid, grp in e.groupby("VariationID", sort=False):
        if len(grp) < 2:
            continue
        recs = grp.to_dict("records")
        for a, b in itertools.combinations(recs, 2):
            if a["Submitter"] != b["Submitter"]:
                cand.append((jaccard(a["pmids"], b["pmids"]),
                             a["Submitter"], b["Submitter"],
                             a["label"] != b["label"]))
    for jt in (0.5, 0.8, 1.0):
        sel = [c for c in cand if c[0] >= jt]
        labs = {" | ".join(sorted([c[1], c[2]])) for c in sel}
        pct = f"{100*sum(c[3] for c in sel)/len(sel):.1f}" if sel else "-"
        print(f"  {mc:>9} {jt:>8.2f} {len(sel):>10,} {len(labs):>10,} {pct:>8}")

bootstrap: code cells 0-3 of are-functional-evidence-claims-in-clinvar-auditabl/__notebook__.ipynb (87 lines of their output held in _bootstrap_log)
1. COVERAGE
  submissions in panel        : 32,562
    labels                    : 1,200 claim  1,495 denial  18 both  29,849 neither
  citing >= 1 PMID            : 4,012 (12.3%)
    labels                    : 692 claim  189 denial  7 both  3,124 neither
  citing >= 3 PMIDs           : 1,849 (5.7%)
  set size where non-zero     : median 2, 90th pct 10, max 57

2. EQUIVALENT-BIBLIOGRAPHY PAIRS  (both >= 3 cites, Jaccard >= 0.80)
  variant-level pairs         : 60  (24 with identical citation sets)
  distinct variants           : 43
  distinct submitter pairs    : 29  -> 2.1 variant-pairs each; largest 22%, top 3 37% of all pairs

3. DIVERGENCE UNDER FIXED EVIDENCE
  variant-level, any difference   :  21.7%   <- inflated by clustering
  variant-level, claim vs denial  :   0.0%
  submitter-pair level, unweighted:  34.9%   (n=29 pairs)
  sub

## 2 · Do claims and denials cite literature at the same rate?

A claim points at a study; a denial has no study to point at, so some gap is expected from the shape
of the sentence alone. The question is whether the corpus-level gap survives *inside* laboratories
that write both kinds of statement — the check that would expose it as one house style counted many
times, which is the same clustering problem as section 1.

Section 2 of the output is therefore the test that can overturn section 1 of it.


In [3]:
"""
Follow-up to the bibliography-overlap diagnostic. Run in the SAME kernel,
after that cell: it reads `work` (one row per submission, with `label`,
`n_cites`, `Submitter`, `VariationID`).

The diagnostic's headline was not the one it was built for. Equivalent-
bibliography pairs turned out too few to support a rate, and its
claim-vs-denial line (0.0%) has no power: denials so rarely cite literature
that ~1 such pair is expected by chance in 60, and seeing zero happens 37% of
the time. What it did surface is that claims and denials differ in whether
they cite anything at all. This cell tests that properly.

Three questions, in the order that could overturn the result:
  1. Does the asymmetry hold on the corpus directly, with intervals?
  2. Does it hold INSIDE laboratories, or is it one lab's template counted
     many times? A house style replicated across thousands of submissions
     produces an aggregate that no individual laboratory shows.
  3. Why does the SOS1 I437T LabCorp/Dasa contrast not appear in the pairs?
"""

import math

import pandas as pd

MIN_PER_LAB = 10          # claims AND denials a lab needs for a within-lab rate
I437T_VARIATION = "45345"


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"),) * 3
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h


if "work" not in globals():
    raise RuntimeError("`work` not in this kernel. Run the bibliography-overlap "
                       "diagnostic cell first, in the same session.")

w = work[work["label"].isin(["claim", "denial"])].copy()
w["cites"] = w["n_cites"] > 0

# --------------------------------------------------------------------------
# 1. Corpus level.
# --------------------------------------------------------------------------
print("=" * 78)
print("1. DOES A CLAIM CITE LITERATURE MORE OFTEN THAN A DENIAL?  (corpus)")
print("=" * 78)
rates = {}
for lab in ("claim", "denial"):
    s = w[w.label == lab]
    k, n = int(s.cites.sum()), len(s)
    p, lo, hi = wilson(k, n)
    rates[lab] = p
    print(f"  {lab:7s}: {k:5,} of {n:5,} cite >= 1 PMID  = {100*p:5.1f}%  "
          f"[95% CI {100*lo:4.1f}-{100*hi:4.1f}]")
print(f"\n  ratio claim/denial: {rates['claim']/rates['denial']:.2f}x")
print("  (both-flagged submissions excluded; 'neither' is not a claim about "
      "evidence at all)")

# --------------------------------------------------------------------------
# 2. Within laboratories. The test that could overturn section 1.
# --------------------------------------------------------------------------
per = (w.groupby(["Submitter", "label"])["cites"]
         .agg(["sum", "size"]).unstack("label"))
per.columns = [f"{a}_{b}" for a, b in per.columns]
for c in ("sum_claim", "size_claim", "sum_denial", "size_denial"):
    if c not in per.columns:
        per[c] = 0
per = per.fillna(0)
both_sides = per[(per.size_claim >= MIN_PER_LAB) &
                 (per.size_denial >= MIN_PER_LAB)].copy()
both_sides["cite_rate_claim"] = both_sides.sum_claim / both_sides.size_claim
both_sides["cite_rate_denial"] = both_sides.sum_denial / both_sides.size_denial
both_sides["diff_pp"] = 100 * (both_sides.cite_rate_claim -
                               both_sides.cite_rate_denial)
both_sides = both_sides.sort_values("size_claim", ascending=False)

print()
print("=" * 78)
print(f"2. WITHIN LABORATORIES  (labs with >= {MIN_PER_LAB} claims AND "
      f">= {MIN_PER_LAB} denials)")
print("=" * 78)
if both_sides.empty:
    print("  no laboratory submits enough of both. The asymmetry cannot be")
    print("  tested within labs, so section 1 may be a between-lab effect -")
    print("  claims and denials written by different laboratories with")
    print("  different citation habits. Report it that way, not as a")
    print("  property of how evidence is asserted.")
else:
    show = both_sides[["size_claim", "cite_rate_claim",
                       "size_denial", "cite_rate_denial", "diff_pp"]]
    print(show.to_string(float_format=lambda x: f"{x:.3f}"))
    n_labs = len(both_sides)
    n_pos = int((both_sides.diff_pp > 0).sum())
    print(f"\n  labs where claims cite more often than denials : "
          f"{n_pos} of {n_labs}")
    print(f"  median within-lab difference                   : "
          f"{both_sides.diff_pp.median():+.1f} percentage points")

    # How much of the corpus-level gap do these labs carry, and does the gap
    # survive removing the single largest contributor?
    top = per.size_claim.add(per.size_denial).idxmax()
    rest = w[w.Submitter != top]
    rc = rest[rest.label == "claim"].cites.mean()
    rd = rest[rest.label == "denial"].cites.mean()
    print(f"\n  largest contributor overall                    : {top}")
    print(f"  corpus ratio WITHOUT it                        : "
          f"{rc/rd:.2f}x  (claim {100*rc:.1f}% vs denial {100*rd:.1f}%)")

cover = w.Submitter.isin(both_sides.index).mean() if len(both_sides) else 0
print(f"\n  share of claim+denial submissions from labs tested here: "
      f"{100*cover:.1f}%")
print("  The rest come from labs that assert only one side - which is itself")
print("  part of the answer: much of the asymmetry may be WHO writes claims")
print("  versus who writes denials, not how either is written.")

# --------------------------------------------------------------------------
# 3. The worked example.
# --------------------------------------------------------------------------
print()
print("=" * 78)
print(f"3. SOS1 p.Ile437Thr  (VariationID {I437T_VARIATION})")
print("=" * 78)
v = work[work.VariationID.astype(str) == I437T_VARIATION]
if v.empty:
    print("  not in this panel's submissions.")
else:
    cols = ["Submitter", "label", "n_cites", "DateLastEvaluated"]
    print(v[cols].sort_values("n_cites", ascending=False)
                 .to_string(index=False))
    in_pairs = (pairs.VariationID.astype(str) == I437T_VARIATION).sum() \
        if "pairs" in globals() and not pairs.empty else 0
    print(f"\n  equivalent-bibliography pairs from this variant: {in_pairs}")
    print("  A submitter with 0 citations never enters the pairing, whatever")
    print("  its label. If LabCorp's denial cites nothing, then LabCorp x Dasa")
    print("  is a real contradiction on this record but NOT a comparison under")
    print("  fixed evidence - only GeneDx x Dasa is.")

1. DOES A CLAIM CITE LITERATURE MORE OFTEN THAN A DENIAL?  (corpus)
  claim  :   692 of 1,200 cite >= 1 PMID  =  57.7%  [95% CI 54.9-60.4]
  denial :   189 of 1,495 cite >= 1 PMID  =  12.6%  [95% CI 11.1-14.4]

  ratio claim/denial: 4.56x
  (both-flagged submissions excluded; 'neither' is not a claim about evidence at all)

2. WITHIN LABORATORIES  (labs with >= 10 claims AND >= 10 denials)
                                                                            size_claim  cite_rate_claim  size_denial  cite_rate_denial  diff_pp
Submitter                                                                                                                                      
Labcorp Genetics (formerly Invitae), Labcorp                                   228.000            1.000      282.000             0.074   92.553
Ambry Genetics                                                                 171.000            0.000       61.000             0.000    0.000
GeneDx                         

## 3 · The worked example — SOS1 c.1310T>C (p.Ile437Thr)

Which PMIDs each submitter names in its comment text, and how far the sets overlap.

Note what this cell can and cannot see. `submission_summary.txt` carries only the free-text comment.
ClinVar also stores references in a structured citation field, which the record page merges in and
this file does not. For this variant the difference is material: the record page lists 11 references
for one submitter whose comment text names 5. Any conclusion here is about comment text, not about
the record.


In [4]:
"""
Which references does each submitter on SOS1 p.Ile437Thr (VariationID 45345)
name in its comment text, and how do the sets overlap?

Run in the same kernel as the bibliography-overlap diagnostic (reads `work`).

Why this is needed: the ClinVar web page lists 11 PMIDs for both GeneDx and
Dasa, but the pipeline, which reads only the free-text comment in
submission_summary.txt, sees 11 for GeneDx, 5 for Dasa and 3 for LabCorp. The
web page merges the comment with the submission's structured citation field;
the tab-delimited file carries only the comment. This prints what the pipeline
sees, which is the only view a text-mining study of ClinVar can rely on.
"""

import itertools

VID = "45345"

# The 11 PMIDs the ClinVar web page lists for both GeneDx and Dasa.
RECORD_11 = {"12628188", "17143282", "20648242", "21387466", "24451042",
             "24803665", "29493581", "30039904", "30541462", "31785789",
             "33042901"}

v = work[work.VariationID.astype(str) == VID]
sets = {}
for _, r in v.iterrows():
    if len(r["pmids"]):
        sets[r["Submitter"]] = (r["label"], set(r["pmids"]))

print(f"SOS1 p.Ile437Thr, VariationID {VID}: PMIDs named in the COMMENT TEXT\n")
for sub, (lab, s) in sorted(sets.items(), key=lambda kv: -len(kv[1][1])):
    outside = s - RECORD_11
    print(f"{sub[:62]:62s} {lab:8s} n={len(s):2d}")
    print(f"    {sorted(s)}")
    if outside:
        print(f"    not among the 11 on the record page: {sorted(outside)}")
    print()

print("pairwise overlap (Jaccard | shared PMIDs):")
for (a, (la, sa)), (b, (lb, sb)) in itertools.combinations(sets.items(), 2):
    inter = sa & sb
    j = len(inter) / len(sa | sb)
    print(f"  {a[:30]:30s} x {b[:30]:30s}  J={j:.2f}  "
          f"shared={len(inter)}  labels {la}/{lb}")

SOS1 p.Ile437Thr, VariationID 45345: PMIDs named in the COMMENT TEXT

GeneDx                                                         neither  n=11
    ['12628188', '17143282', '20648242', '21387466', '24451042', '24803665', '29493581', '30039904', '30541462', '31785789', '33042901']

Dasa                                                           claim    n= 5
    ['21387466', '24451042', '24803665', '30039904', '30541462']

Women's Health and Genetics/Laboratory Corporation of America, denial   n= 3
    ['21387466', '22585553', '30039904']
    not among the 11 on the record page: ['22585553']

Center for Genomics, Ann and Robert H. Lurie Children's Hospit neither  n= 3
    ['21387466', '30039904', '30541462']

3billion                                                       neither  n= 3
    ['21387466', '24451042', '29493581']

Labcorp Genetics (formerly Invitae), Labcorp                   neither  n= 2
    ['21387466', '24451042']

Victorian Clinical Genetics Services, Murdoch Children

## 4 · Which sentence is each citation attached to?

A PMID appearing somewhere in a submission is not the same as being cited *for* that submission's
claim or denial — a comment that denies functional evidence will often cite case reports in a
different sentence. This cell attributes each PMID to the sentence it sits in, and classifies that
sentence with the same per-sentence logic the published classifier uses.

**Section 0 of the output is the control.** The sentence-level roles must reproduce the published
claim/denial flags exactly on every citing submission, or nothing below can be trusted. They do:
4,012 of 4,012.

Section 3 is the structural reason the strict count is near zero: of roughly 18,000 sentence-level
citations, almost none sit in a sentence that denies functional evidence. A denial rarely cites
anything, so a design requiring both sides to cite the same paper can only ever see the few
laboratories whose template lists the literature they reviewed.


In [5]:
"""
Sentence-level citation attribution: in which sentence is each PMID cited,
and what does that sentence assert?

Run in the same kernel as the bibliography-overlap diagnostic. Needs `work`
and the classifier's own regexes from panel_functional_claims_v6 / the Kaggle
notebook: _mend, _SENT, VENDOR_MODEL, CONDITIONAL, DENIAL, CLAIM, CODE_PS3.

WHY SENTENCE LEVEL
    Whole-bibliography overlap (Jaccard) was the wrong unit: on SOS1 I437T it
    missed the case entirely. The sharper unit is the single citation - one
    paper, cited on one variant, read one way by one submitter and another way
    by another. But a PMID being somewhere in a submission is not the same as
    being cited FOR that submission's claim or denial: a comment that denies
    functional evidence will often cite case reports in a different sentence.
    So each PMID is attributed to the sentence it sits in, and the sentence is
    classified with exactly the per-sentence logic classify() uses.

SENTENCE SPLITTING
    Uses v6's own _SENT, which splits after '.' or ';' before an uppercase
    letter or '('. That orphans citations written as "... effect. (PMID: 1)" or
    "...; PMID: 1": the fragment becomes its own sentence. A fragment holding
    nothing but a citation is attached to the sentence before it.

RECONCILIATION
    Section 0 re-derives each submission's claim/denial flags from the
    sentence roles and compares them with `work`. Anything below 100% agreement
    means this cell and the published classifier read some submissions
    differently, and those rows are counted before any result is shown.
"""

import itertools
import re
from collections import defaultdict

import pandas as pd

VID_EXAMPLE = "45345"

need = ["work", "_mend", "_SENT", "VENDOR_MODEL", "CONDITIONAL", "DENIAL",
        "CLAIM", "CODE_PS3", "cited_pmids"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"not in this kernel: {', '.join(missing)}. Run the "
                       "classifier notebook's first cells and the "
                       "bibliography-overlap diagnostic first.")

CITE_ONLY = re.compile(
    r"^\s*\(?\s*(?:PMIDs?\s*:?\s*)?[\d\s,;:&/and()PMID.]*\)?\s*[.;]?\s*$", re.I)


def sentence_roles(text):
    """[(role, frozenset(pmids), sentence)] mirroring classify()'s logic."""
    t = _mend(str(text or ""))
    raw = [s for s in _SENT.split(t) if s.strip()]
    merged = []
    for s in raw:
        if merged and cited_pmids(s) and CITE_ONLY.match(s):
            merged[-1] = merged[-1] + " " + s
        else:
            merged.append(s)
    out = []
    for s in merged:
        pm = cited_pmids(s)
        if VENDOR_MODEL.search(s):
            role = "vendor-model"
        elif CONDITIONAL.search(s):
            role = "conditional"
        else:
            d = bool(DENIAL.search(s))
            c = (bool(CLAIM.search(s)) or bool(CODE_PS3.search(s))) and not d
            role = "denial" if d else ("claim" if c else "other")
        out.append((role, pm, s))
    return out


# --------------------------------------------------------------------------
# 0. Reconciliation against the published flags.
# --------------------------------------------------------------------------
cit = work[work.n_cites > 0].copy()
cit["roles"] = cit["text"].map(sentence_roles)
cit["claim_s"] = cit.roles.map(lambda rs: any(r == "claim" for r, _, _ in rs))
cit["denial_s"] = cit.roles.map(lambda rs: any(r == "denial" for r, _, _ in rs))
agree = ((cit.claim_s == cit.claim.astype(bool)) &
         (cit.denial_s == cit.denial.astype(bool)))

print("=" * 78)
print("0. RECONCILIATION  (sentence roles vs published classify() flags)")
print("=" * 78)
print(f"citing submissions                     : {len(cit):,}")
print(f"  flags reproduced exactly             : {int(agree.sum()):,} "
      f"({100*agree.mean():.2f}%)")
if (~agree).any():
    print(f"  disagree                             : {int((~agree).sum()):,}"
          "  - these read differently here than in the published run;")
    print("    the vendor-model sentence is the likely source (classify() strips")
    print("    it by span before splitting, this cell tests it per sentence).")

# long form: one row per (submission, PMID, sentence role)
rows = []
for _, r in cit.iterrows():
    attached = set()
    for role, pm, _s in r["roles"]:
        for p in pm:
            rows.append(dict(VariationID=str(r["VariationID"]), gene=r["gene"],
                             Submitter=r["Submitter"], pmid=p,
                             sent_role=role, sub_label=r["label"]))
            attached.add(p)
    for p in set(r["pmids"]) - attached:          # parsed from the joined text
        rows.append(dict(VariationID=str(r["VariationID"]), gene=r["gene"],
                         Submitter=r["Submitter"], pmid=p,
                         sent_role="unattached", sub_label=r["label"]))
cites = (pd.DataFrame(rows)
           .drop_duplicates(["VariationID", "Submitter", "pmid", "sent_role"]))

# --------------------------------------------------------------------------
# 1. The worked example.
# --------------------------------------------------------------------------
print()
print("=" * 78)
print(f"1. SOS1 p.Ile437Thr (VariationID {VID_EXAMPLE}): the sentence each "
      "PMID sits in")
print("=" * 78)
ex = cit[cit.VariationID.astype(str) == VID_EXAMPLE]
for _, r in ex.iterrows():
    print(f"\n{r['Submitter'][:70]}   [submission: {r['label']}]")
    for role, pm, s in r["roles"]:
        if pm or role in ("claim", "denial"):
            flat = " ".join(s.split())
            print(f"  {role:12s} {str(sorted(pm)) if pm else '(no citation)'}")
            print(f"      \"{flat[:150]}{'...' if len(flat) > 150 else ''}\"")

# --------------------------------------------------------------------------
# 2. Panel-wide: one paper, one variant, read differently.
# --------------------------------------------------------------------------
cell = cites.groupby(["VariationID", "pmid"])
multi = cell.Submitter.nunique()
multi = multi[multi >= 2]

def cell_flags(g):
    subs_claim_sent = set(g.loc[g.sent_role == "claim", "Submitter"])
    subs_den_sent = set(g.loc[g.sent_role == "denial", "Submitter"])
    subs_den_sub = set(g.loc[g.sub_label.isin(["denial", "both"]), "Submitter"])
    strict = any(a != b for a in subs_claim_sent for b in subs_den_sent)
    loose = any(a != b for a in subs_claim_sent for b in subs_den_sub)
    return pd.Series(dict(strict=strict, loose=loose,
                          claim_sent=len(subs_claim_sent),
                          den_sub=len(subs_den_sub)))

keyed = cites.set_index(["VariationID", "pmid"]).loc[multi.index].reset_index()
flags = (keyed.groupby(["VariationID", "pmid"])[["Submitter", "sent_role",
                                                  "sub_label"]]
              .apply(cell_flags))

print()
print("=" * 78)
print("2. ONE PAPER, ONE VARIANT, TWO READINGS  (panel-wide)")
print("=" * 78)
print(f"(variant, PMID) cells cited by >= 2 submitters     : {len(flags):,}")
print(f"  ...where some submitter cites it in a CLAIM sentence: "
      f"{int((flags.claim_sent > 0).sum()):,}")
print(f"  ...where some DENIAL-labelled submission cites it : "
      f"{int((flags.den_sub > 0).sum()):,}   <- ceiling on both counts below")
print()
print(f"claim sentence  x  denial SENTENCE, same paper     : "
      f"{int(flags.strict.sum()):,} cells  (strict)")
print(f"claim sentence  x  denial SUBMISSION, same paper   : "
      f"{int(flags.loose.sum()):,} cells  (loose)")

for name, col in (("strict", "strict"), ("loose", "loose")):
    hit = flags[flags[col]].reset_index()
    if hit.empty:
        continue
    k = keyed.merge(hit[["VariationID", "pmid"]], on=["VariationID", "pmid"])
    pairs_ = set()
    for (vid, p), g in k.groupby(["VariationID", "pmid"]):
        a = set(g.loc[g.sent_role == "claim", "Submitter"])
        b = (set(g.loc[g.sent_role == "denial", "Submitter"]) if col == "strict"
             else set(g.loc[g.sub_label.isin(["denial", "both"]), "Submitter"]))
        for x, y in itertools.product(a, b):
            if x != y:
                pairs_.add(" | ".join(sorted([x, y])))
    print(f"\n  {name}: {hit.VariationID.nunique():,} variants, "
          f"{hit.pmid.nunique():,} distinct papers, "
          f"{len(pairs_):,} distinct submitter pairs")
    top_p = hit.pmid.value_counts().head(8)
    print(f"  papers most often read both ways ({name}):")
    for p, n in top_p.items():
        print(f"    PMID {p}: {n} variant(s)")

cites.to_csv("citations_by_sentence.csv", index=False)
flags.reset_index().to_csv("paper_variant_readings.csv", index=False)
print("\nwrote citations_by_sentence.csv, paper_variant_readings.csv")

# --------------------------------------------------------------------------
# 3. Where citations sit, by what the sentence asserts. If denial sentences
#    almost never carry a PMID, the strict count above is bounded near zero by
#    writing convention, not by agreement between laboratories.
# --------------------------------------------------------------------------
print()
print("=" * 78)
print("3. WHICH SENTENCES CARRY CITATIONS")
print("=" * 78)
print(cites.sent_role.value_counts().to_string())

0. RECONCILIATION  (sentence roles vs published classify() flags)
citing submissions                     : 4,012
  flags reproduced exactly             : 4,012 (100.00%)

1. SOS1 p.Ile437Thr (VariationID 45345): the sentence each PMID sits in

Dasa   [submission: claim]
  claim        ['21387466', '24451042', '24803665', '30039904', '30541462']
      "Functional evidence supports a deleterious effect on the gene or gene product (PMID: 21387466; PMID: 24803665; PMID: 30541462; PMID: 30039904; PMID: 2..."
  other        ['21387466', '24451042', '24803665', '30039904', '30541462']
      "This variant has been recurrently observed in individuals with related phenotype (PMID: 21387466; PMID: 24803665; PMID: 30541462; PMID: 30039904; PMID..."

Labcorp Genetics (formerly Invitae), Labcorp   [submission: neither]
  other        ['21387466', '24451042']
      "This missense change has been observed in individual(s) with clinical features of Noonan syndrome (PMID: 21387466, 24451042)."

GeneDx  

## 5 · Reading the flagged cases

A mechanical search is not a case series until each case has been read. This cell prints every
flagged variant sentence by sentence, with the three questions each has to pass: does the claim
sentence concern *this* variant's function; is the denial about *this* variant's function; and do
both submitters cite the paper as evidence about this variant rather than for population data, gene
level mechanism, or a different allele?

Of the four flagged, **two hold**. The two that fail do so because the search counted submissions
labelled *both* claim and denial on the denial side: in each, one laboratory asserts experimental
evidence in one sentence and carries a standard caveat about in-silico predictions in another. A
count without reading would have reported four.


In [6]:
"""
Read the loose-match cases one by one before any count of them is reported.

Run in the same kernel, after citation_sentence_attribution.py (needs `cit`
and `flags`). The addendum states that 4 variants show the SOS1 I437T pattern:
one submitter cites a paper in a sentence asserting functional evidence, while
another cites the same paper in a submission that denies it. That number came
from a mechanical count and only I437T has been read. A count is not a case
series until each case has been looked at.

For each variant this prints the ClinVar link, and for every submitter that
cites one of the flagged papers: the sentence carrying the paper, plus - for
denial submissions - the denial sentence itself, so the two can be compared.
"""

need = ["cit", "flags"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"not in this kernel: {', '.join(missing)}. Run "
                       "citation_sentence_attribution.py first.")

hit = flags[flags["loose"]].reset_index()
print(f"{hit.VariationID.nunique()} variants, {len(hit)} (variant, paper) cells\n")

for vid, g in hit.groupby("VariationID"):
    papers = set(g.pmid)
    rows = cit[cit.VariationID.astype(str) == str(vid)]
    gene = rows.gene.iloc[0] if len(rows) else "?"
    print("=" * 78)
    print(f"{gene}  VariationID {vid}   flagged papers: {sorted(papers)}")
    print(f"https://www.ncbi.nlm.nih.gov/clinvar/variation/{vid}/")
    print("=" * 78)
    for _, r in rows.iterrows():
        rel = [(role, pm & papers, s) for role, pm, s in r["roles"]
               if (pm & papers) or role == "denial"]
        if not any(pm for _, pm, _ in rel):
            continue
        print(f"\n  {r['Submitter'][:70]}   [submission: {r['label']}]"
              f"   {r.get('DateLastEvaluated', '')}")
        for role, pm, s in rel:
            flat = " ".join(s.split())
            tag = str(sorted(pm)) if pm else "(no flagged paper)"
            print(f"    {role:9s} {tag}")
            print(f"        \"{flat[:260]}{'...' if len(flat) > 260 else ''}\"")
    print()

print("For each variant, check three things before counting it:")
print("  1. the claim sentence says functional evidence exists FOR THIS variant;")
print("  2. the denial is about THIS variant's function, not another criterion;")
print("  3. both submissions cite the paper as evidence about this variant,")
print("     not for population data, gene-level mechanism, or a different allele.")

4 variants, 8 (variant, paper) cells

PTPN11  VariationID 13325   flagged papers: ['11704759', '12960218', '15956085']
https://www.ncbi.nlm.nih.gov/clinvar/variation/13325/

  Women's Health and Genetics/Laboratory Corporation of America, LabCorp   [submission: denial]   Jun 16, 2025
    denial    (no flagged paper)
        "To our knowledge, no experimental evidence demonstrating an impact on protein function has been reported."
    other     ['11704759', '12960218', '15956085']
        "The following publications have been ascertained in the context of this evaluation (PMID: 32561839, 27069254, 19179468, 22465605, 15956085, 17972951, 15928039, 21784453, 14644997, 15385933, 15710330, 27276561, 12960218, 25395418, 25097206, 11704759, 12717436, ..."

  Labcorp Genetics (formerly Invitae), Labcorp   [submission: neither]   Feb 17, 2025
    other     ['11704759', '12960218', '15956085']
        "This missense change has been observed in individual(s) with Noonan syndrome and hypertrophic 

## 6 · Robustness — do the pipeline's own defects change the answer?

Two independent code reviews of the pipeline behind the published counts found four defects: a gene
filter that drops submissions filed with an empty gene field, a sentence splitter that does not break
at bullet markers, negated sentences read as claims, and a denial phrasing the classifier misses.

This cell measures each one **separately against the published counts**, rather than correcting them
first. A and B are recomputations and can be quoted. C and D are pattern-based estimates: they write
their candidates to CSV, and those have to be read before any number from them is used — section 5
above is the same discipline applied to a different count.


In [7]:
"""
How much do the defects found by the two independent reviews change the
PUBLISHED panel counts? Measure before correcting anything.

Run at the end of the classifier notebook (are-functional-evidence-claims-in-
clinvar-auditabl), in the same kernel, after its first four code cells. It
needs `subs`, `classify`, the classifier's regexes and `_blank`, and re-reads
the cached submission_summary.txt.gz; nothing is downloaded.

Four defects, each measured separately against the published counts:

  A. load_panel keeps a row only if ITS gene field names a panel gene, so
     submissions filed with gene "-" on panel variants are dropped.
     (Found by review 1.) Variants on which EVERY submission has gene "-" are
     not recoverable this way and remain missing.
  B. _SENT splits only before a capital letter or "(", so "... . - No
     published functional evidence ..." stays one sentence, and inside
     classify() a denial then suppresses a claim in the same span.
     (Review 1.) Measured by re-running classify() with a splitter that also
     breaks at bullets; the unchanged splitter must first reproduce the
     published flags exactly, or the comparison means nothing.
  C. Negated or prospective sentences read as claims: "No prior functional
     evidence has been identified", "PS3 was not applied", "A zebrafish model
     is needed". (Review 2.) A submission is counted only if EVERY sentence
     that produced its claim carries such a cue. Tier 1 uses three narrow
     patterns; tier 2 any negation word. Both are candidates to READ, not
     corrections: "functional studies showed the variant did not bind" is a
     genuine claim with "not" in it.
  D. Missed denials such as "Functional studies have not yet been performed".
     (Review 2.) One narrow pattern, qualifier REQUIRED: the bare "has not
     been reported" is the literature-absence phrase that broke version 1 of
     this classifier and must not come back.

Every candidate is written to CSV with the sentence that triggered it.
"""

import csv
import glob
import re
from pathlib import Path

import pandas as pd

need = ["subs", "classify", "CLAIM", "DENIAL", "CODE_PS3", "CONDITIONAL",
        "VENDOR_MODEL", "_SENT", "_mend", "_blank", "read_header", "WANTED",
        "PANEL"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"not in this kernel: {', '.join(missing)}. Run the "
                       "classifier notebook's first four code cells first.")

OUTDIR = Path("published_pipeline_impact")
OUTDIR.mkdir(exist_ok=True)


def counts(claim, denial):
    claim, denial = pd.Series(claim).astype(bool), pd.Series(denial).astype(bool)
    return dict(claims=int(claim.sum()), denials=int(denial.sum()),
                both=int((claim & denial).sum()))


def classify_with(text, splitter):
    """classify()'s claim/denial logic with the sentence splitter swapped."""
    t = _mend(str(text or ""))
    t = VENDOR_MODEL.sub(" ", t)
    claim = denial = False
    for s in splitter.split(t):
        if not s or not s.strip() or CONDITIONAL.search(s):
            continue
        d = bool(DENIAL.search(s))
        c = (bool(CLAIM.search(s)) or bool(CODE_PS3.search(s))) and not d
        claim, denial = claim or c, denial or d
    return claim, denial


def claim_sentences(text):
    """The sentences that produced a claim, as classify() reads them."""
    t = _mend(str(text or ""))
    t = VENDOR_MODEL.sub(" ", t)
    out = []
    for s in _SENT.split(t):
        if not s or not s.strip() or CONDITIONAL.search(s):
            continue
        if DENIAL.search(s):
            continue
        if CLAIM.search(s) or CODE_PS3.search(s):
            out.append(" ".join(s.split()))
    return out


base = counts(subs["claim"], subs["denial"])
rows = [("published", base)]
print("=" * 78)
print("PUBLISHED COUNTS (this kernel)")
print("=" * 78)
print(f"submissions {len(subs):,} | claims {base['claims']:,} | denials "
      f"{base['denials']:,} | both {base['both']:,} | claims/denials "
      f"{base['claims']/max(1, base['denials']):.3f}")

# --------------------------------------------------------------------------
# A. Rows dropped by the per-row gene filter.
# --------------------------------------------------------------------------
path = Path("submission_summary.txt.gz")
if not path.exists():
    hits = glob.glob("/kaggle/**/submission_summary.txt.gz", recursive=True)
    path = Path(hits[0]) if hits else path
print()
print("=" * 78)
print("A. SUBMISSIONS ON PANEL VARIANTS DROPPED BY THE GENE-FIELD FILTER")
print("=" * 78)
if not path.exists():
    print("cached submission_summary.txt.gz not found; section A skipped.")
    dropped = pd.DataFrame()
else:
    skip, names = read_header(path)
    keep = [c for c in WANTED if c in names]
    vids = set(subs["VariationID"].astype(str))
    want = {g.upper() for g in PANEL}
    parts = []
    for ch in pd.read_csv(path, sep="\t", header=None, names=names,
                          skiprows=skip, usecols=keep, dtype=str,
                          quoting=csv.QUOTE_NONE, engine="c", na_filter=False,
                          on_bad_lines="skip", chunksize=200_000):
        sym = ch["SubmittedGeneSymbol"].astype(str).str.strip().str.upper()
        sel = ch["VariationID"].astype(str).isin(vids) & ~sym.isin(want)
        if sel.any():
            parts.append(ch[sel].copy())
    dropped = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if len(dropped):
    txt = _blank(dropped["Description"])
    if "ExplanationOfInterpretation" in dropped.columns:
        txt = txt + " " + _blank(dropped["ExplanationOfInterpretation"])
    dropped["text"] = txt.str.strip()
    fl = dropped["text"].apply(classify).apply(pd.Series)
    dropped["claim"], dropped["denial"] = fl["claim"], fl["denial"]
    a = counts(dropped["claim"], dropped["denial"])
    print(f"dropped rows on panel variants : {len(dropped):,} "
          f"({dropped['VariationID'].nunique():,} variants)")
    print(f"  with a prose comment (>=20)  : {int((dropped['text'].str.len() >= 20).sum()):,}")
    print(f"  claims {a['claims']:,} | denials {a['denials']:,} | both {a['both']:,}")
    print("  gene field on the dropped rows:")
    print(dropped["SubmittedGeneSymbol"].value_counts().head(6).to_string())
    dropped.to_csv(OUTDIR / "A_dropped_rows_classified.csv", index=False)
    rows.append(("A. dropped rows restored",
                 dict(claims=base["claims"] + a["claims"],
                      denials=base["denials"] + a["denials"],
                      both=base["both"] + a["both"])))
else:
    print("no dropped rows found.")

# --------------------------------------------------------------------------
# B. Sentence splitter.
# --------------------------------------------------------------------------
_SENT_FIX = re.compile(r"(?<=[.;])\s+(?=[A-Z(])"
                       r"|(?<=[.;])\s*[-•·�]\s+")
print()
print("=" * 78)
print("B. SPLITTER THAT ALSO BREAKS AT BULLETS  ('. - ', '. • ', '. � ')")
print("=" * 78)
same = subs["text"].apply(lambda t: classify_with(t, _SENT))
repro = ((same.str[0] == subs["claim"].astype(bool)) &
         (same.str[1] == subs["denial"].astype(bool))).mean()
print(f"unchanged splitter reproduces the published flags: {100*repro:.2f}%")
if repro < 1:
    print("  NOT 100%: this cell's copy of the logic differs from classify();")
    print("  section B is not interpretable until that is resolved.")
fixed = subs["text"].apply(lambda t: classify_with(t, _SENT_FIX))
fc, fd = fixed.str[0], fixed.str[1]
flip = (fc != subs["claim"].astype(bool)) | (fd != subs["denial"].astype(bool))
b = counts(fc, fd)
print(f"submissions whose flags change : {int(flip.sum()):,}")
print(f"  claims {base['claims']:,} -> {b['claims']:,} | denials "
      f"{base['denials']:,} -> {b['denials']:,} | both {base['both']:,} -> {b['both']:,}")
flips = subs.loc[flip, ["VariationID", "gene", "Submitter", "claim", "denial",
                        "text"]].copy()
flips["claim_fixed"], flips["denial_fixed"] = fc[flip], fd[flip]
flips.to_csv(OUTDIR / "B_splitter_flips.csv", index=False)
rows.append(("B. bullet-aware splitter", b))

# --------------------------------------------------------------------------
# C. Claims whose every claim sentence is negated or prospective.
# --------------------------------------------------------------------------
TIER1 = re.compile(
    r"\bno\s+(?:\w+\s+){0,3}(?:functional|experimental|in[ -]?vitro|in[ -]?vivo)\b"
    r"|\bPS3\b[^.;]{0,40}\bnot\b|\bnot\b[^.;]{0,20}\bPS3\b"
    r"|\b(?:is|are|would be|will be)\s+(?:needed|required|warranted)\b", re.I)
TIER2 = re.compile(r"\b(?:no|not|nor|without|lack(?:s|ing)?|absent|absence|"
                   r"unknown|unclear|needed|required|yet)\b", re.I)

cl = subs[subs["claim"].astype(bool)].copy()
cl["claim_sents"] = cl["text"].map(claim_sentences)
cl = cl[cl["claim_sents"].map(len) > 0]
cl["tier1"] = cl["claim_sents"].map(lambda ss: all(TIER1.search(s) for s in ss))
cl["tier2"] = cl["claim_sents"].map(lambda ss: all(TIER2.search(s) for s in ss))
n1, n2 = int(cl["tier1"].sum()), int(cl["tier2"].sum())
print()
print("=" * 78)
print("C. CLAIMS WHOSE EVERY CLAIM SENTENCE IS NEGATED OR PROSPECTIVE")
print("=" * 78)
print(f"claim submissions with a claim sentence : {len(cl):,}")
print(f"  tier 1 (three narrow patterns)        : {n1:,}   <- likely false claims")
print(f"  tier 2 (any negation word)            : {n2:,}   <- candidates to read")
for _, r in cl[cl["tier1"]].head(12).iterrows():
    print(f"   [{r['gene']}] {r['Submitter'][:40]}: \"{r['claim_sents'][0][:150]}\"")
out = cl[cl["tier2"]][["VariationID", "gene", "Submitter", "tier1",
                       "claim_sents"]]
out.to_csv(OUTDIR / "C_candidate_false_claims.csv", index=False)
rows.append(("C. minus tier-1 false claims (estimate)",
             dict(claims=base["claims"] - n1, denials=base["denials"],
                  both=base["both"])))

# --------------------------------------------------------------------------
# D. Missed denials, one narrow pattern.
# --------------------------------------------------------------------------
MISSED_DENIAL = re.compile(
    r"\b(?:functional|experimental|in[ -]?vitro|in[ -]?vivo)\s+"
    r"(?:studies|data|evidence|assays?|analys[ie]s|experiments)\s+"
    r"(?:have|has)\s+not\s+(?:yet\s+)?been\s+"
    r"(?:performed|conducted|done|carried out|reported)\b", re.I)
nd = subs[~subs["denial"].astype(bool)].copy()
nd["hit"] = nd["text"].map(lambda t: bool(MISSED_DENIAL.search(str(t))))
n_d = int(nd["hit"].sum())
print()
print("=" * 78)
print("D. MISSED DENIALS  ('<functional> studies have not (yet) been performed')")
print("=" * 78)
print(f"submissions not flagged as denial that match : {n_d:,}")
for _, r in nd[nd["hit"]].head(8).iterrows():
    m = MISSED_DENIAL.search(r["text"])
    s = r["text"][max(0, m.start() - 60):m.end() + 40]
    print(f"   [{r['gene']}] {r['Submitter'][:40]}: \"...{' '.join(s.split())}...\"")
nd[nd["hit"]][["VariationID", "gene", "Submitter", "claim", "text"]] \
    .to_csv(OUTDIR / "D_candidate_missed_denials.csv", index=False)
rows.append(("D. plus missed denials (estimate)",
             dict(claims=base["claims"], denials=base["denials"] + n_d,
                  both=base["both"])))

# --------------------------------------------------------------------------
# Summary. A-B are recomputations; C-D are pattern-based estimates until the
# CSVs have been read.
# --------------------------------------------------------------------------
print()
print("=" * 78)
print("SUMMARY  (each row changes ONE thing relative to the published counts)")
print("=" * 78)
print(f"{'':40s} {'claims':>8s} {'denials':>8s} {'both':>6s} {'ratio':>7s}")
for name, c in rows:
    print(f"{name:40s} {c['claims']:>8,} {c['denials']:>8,} {c['both']:>6,} "
          f"{c['claims']/max(1, c['denials']):>7.3f}")
print()
print("A and B are recomputations. C and D are pattern-based estimates: read")
print(f"the CSVs in {OUTDIR}/ before quoting any of them.")

PUBLISHED COUNTS (this kernel)
submissions 32,562 | claims 1,218 | denials 1,513 | both 18 | claims/denials 0.805

A. SUBMISSIONS ON PANEL VARIANTS DROPPED BY THE GENE-FIELD FILTER
dropped rows on panel variants : 4,007 (2,737 variants)
  with a prose comment (>=20)  : 354
  claims 111 | denials 23 | both 0
  gene field on the dropped rows:
SubmittedGeneSymbol
-              3998
HRAS;LRRC56       2
MKRN2;RAF1        1
RIT               1
LZTR1;THAP7       1
SOX17             1

B. SPLITTER THAT ALSO BREAKS AT BULLETS  ('. - ', '. • ', '. � ')
unchanged splitter reproduces the published flags: 100.00%
submissions whose flags change : 0
  claims 1,218 -> 1,218 | denials 1,513 -> 1,513 | both 18 -> 18

C. CLAIMS WHOSE EVERY CLAIM SENTENCE IS NEGATED OR PROSPECTIVE
claim submissions with a claim sentence : 1,218
  tier 1 (three narrow patterns)        : 17   <- likely false claims
  tier 2 (any negation word)            : 100   <- candidates to read
   [PTPN11] GeneDx: "Published function

## 7 · The robustness result

Everything below is pure HTML and SVG with no charting library: the data sits in two constants at the
top (`RUNS` and `CAUSES`), and a little JavaScript underneath draws the geometry. Hover any mark for
its value; each figure carries a table view.


In [8]:
"""
Figures for section 7. Pure HTML and SVG, no charting library.

The two constants at the top of the <script> block hold the measured values:
  RUNS   - the five recomputation scenarios, with claims, denials and the ratio
  CAUSES - the four defects, with the change each makes to the ratio

Everything else is geometry. The values are transcribed from the section 6 output
above so the figures render without re-running the pipeline; the table view under
each figure carries the same numbers for checking.
"""

from IPython.display import HTML, display

FIGURES = r"""
<div class="viz-root" id="viz">
<style>
.viz-root{
  color-scheme: light;
  --surface-1:#fcfcfb; --text-primary:#0b0b0b; --text-secondary:#52514e;
  --text-muted:#898781; --grid:#e1e0d9; --axis:#c3c2b7;
  --claims:#2a78d6; --denials:#eb6834; --up:#2a78d6; --down:#e34948; --mid:#f0efec;
  background:var(--surface-1); color:var(--text-primary);
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Helvetica,Arial,sans-serif;
  padding:4px 0 10px; max-width:920px;
}
@media (prefers-color-scheme: dark){
  :root:where(:not([data-theme="light"])) .viz-root{
    color-scheme: dark;
    --surface-1:#1a1a19; --text-primary:#ffffff; --text-secondary:#c3c2b7;
    --text-muted:#898781; --grid:#2c2c2a; --axis:#383835;
    --claims:#3987e5; --denials:#d95926; --up:#3987e5; --down:#e66767; --mid:#383835;
  }
}
:root[data-theme="dark"] .viz-root{
  color-scheme: dark;
  --surface-1:#1a1a19; --text-primary:#ffffff; --text-secondary:#c3c2b7;
  --text-muted:#898781; --grid:#2c2c2a; --axis:#383835;
  --claims:#3987e5; --denials:#d95926; --up:#3987e5; --down:#e66767; --mid:#383835;
}
.viz-root h3{font-size:15px;margin:18px 0 2px;font-weight:600;color:var(--text-primary)}
.viz-root p.sub{font-size:12.5px;margin:0 0 10px;color:var(--text-secondary);max-width:760px;line-height:1.45}
.viz-root .lg{display:flex;gap:16px;align-items:center;font-size:12px;color:var(--text-secondary);margin:0 0 6px}
.viz-root .lg i{display:inline-block;width:11px;height:11px;border-radius:2px;margin-right:5px;vertical-align:-1px}
.viz-root details{margin-top:8px;font-size:12px;color:var(--text-secondary)}
.viz-root summary{cursor:pointer;color:var(--text-muted)}
.viz-root table{border-collapse:collapse;margin-top:8px;font-size:12px}
.viz-root th,.viz-root td{padding:3px 12px 3px 0;text-align:left;border-bottom:1px solid var(--grid);white-space:nowrap}
.viz-root th{color:var(--text-muted);font-weight:600}
.viz-tip{position:fixed;pointer-events:none;opacity:0;transition:opacity .08s;
  background:var(--text-primary);color:var(--surface-1);font-size:11.5px;line-height:1.5;
  padding:6px 9px;border-radius:5px;z-index:9999;white-space:nowrap}
</style>

<h3>Does any pipeline defect overturn the published result?</h3>
<p class="sub">Each row recomputes the panel counts changing <b>one</b> thing. The ratio stays below parity in
every scenario: denials outnumber claims throughout. Scenarios A and B are full recomputations; C is a
pattern-based upper bound that over-counts, and is shown as such.</p>
<div class="lg"><span><i style="background:var(--claims)"></i>claims</span><span><i style="background:var(--denials)"></i>denials</span></div>
<div id="f1"></div>
<details><summary>data table</summary><div id="t1"></div></details>

<h3>Where the change comes from</h3>
<p class="sub">Change in the claims/denials ratio attributable to each defect, measured separately against the
published value of 0.805. Two of the four move nothing at all.</p>
<div id="f2"></div>
<details><summary>data table</summary><div id="t2"></div></details>

<script>
// ---------------------------------------------------------------- data
const RUNS = [
  {id:"published", label:"published",            note:"as released",                       claims:1218, denials:1513, ratio:0.805, kind:"base"},
  {id:"A",         label:"A · gene filter",      note:"dropped rows restored",             claims:1329, denials:1536, ratio:0.865, kind:"recompute"},
  {id:"B",         label:"B · sentence splitter",note:"bullet-aware split",                claims:1218, denials:1513, ratio:0.805, kind:"recompute"},
  {id:"C",         label:"C · negated claims",   note:"tier-1 removed — upper bound",      claims:1201, denials:1513, ratio:0.794, kind:"estimate"},
  {id:"D",         label:"D · missed denials",   note:"none found",                        claims:1218, denials:1513, ratio:0.805, kind:"estimate"}
];

const CAUSES = [
  {id:"A", label:"Gene-field filter",   detail:"4,007 rows on panel variants dropped; 354 with prose. The dropped set is claim-rich (111 vs 23).", delta:+0.060, measured:true},
  {id:"B", label:"Sentence splitter",   detail:"Splitting at bullets as well changes 0 of 32,562 submissions.",                                    delta: 0.000, measured:true},
  {id:"C", label:"Negated claim text",  detail:"Tier-1 pattern flags 17; reading the printed examples, most are genuine claims with a caveat.",    delta:-0.011, measured:false},
  {id:"D", label:"Missed denial text",  detail:"'<functional> studies have not been performed' matches 0 unflagged submissions.",                  delta: 0.000, measured:true}
];

// ------------------------------------------------------------- helpers
const NS="http://www.w3.org/2000/svg";
const el=(n,a={})=>{const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);return e;};
const txt=(x,y,s,o={})=>{const t=el("text",Object.assign({x,y},o));t.textContent=s;return t;};
const fmt=n=>n.toLocaleString("en-US");

// bar with a 4px rounded data-end, square at the baseline
function bar(x0,y,w,h,fill,r=4){
  if(w<=0.5) return el("g");
  const rr=Math.min(r,w);
  const d=`M${x0},${y} H${x0+w-rr} A${rr},${rr} 0 0 1 ${x0+w},${y+rr} V${y+h-rr} A${rr},${rr} 0 0 1 ${x0+w-rr},${y+h} H${x0} Z`;
  return el("path",{d,fill});
}
// mirrored for bars growing left from a zero baseline
function barL(x0,y,w,h,fill,r=4){
  if(w<=0.5) return el("g");
  const rr=Math.min(r,w);
  const d=`M${x0},${y} H${x0-w+rr} A${rr},${rr} 0 0 0 ${x0-w},${y+rr} V${y+h-rr} A${rr},${rr} 0 0 0 ${x0-w+rr},${y+h} H${x0} Z`;
  return el("path",{d,fill});
}

// one shared tooltip
const tip=document.createElement("div"); tip.className="viz-tip"; document.body.appendChild(tip);
function hover(node,html){
  node.style.cursor="default";
  node.addEventListener("mousemove",e=>{tip.innerHTML=html;tip.style.opacity=1;
    tip.style.left=Math.min(e.clientX+14,innerWidth-tip.offsetWidth-8)+"px";
    tip.style.top=(e.clientY-tip.offsetHeight-10)+"px";});
  node.addEventListener("mouseleave",()=>tip.style.opacity=0);
}

// ------------------------------------------------- figure 1 · scenarios
(function(){
  const W=900, ROW=56, TOP=42, LAB=178,
        CX0=LAB+10, CX1=560, RX0=642, RX1=868,
        H=TOP+RUNS.length*ROW+40;
  const cmax=1700, rmax=1.10;
  const cx=v=>CX0+(v/cmax)*(CX1-CX0);
  const rx=v=>RX0+(v/rmax)*(RX1-RX0);
  const svg=el("svg",{width:"100%",viewBox:`0 0 ${W} ${H}`,role:"img",
    "aria-label":"Claims and denials per scenario, and the claims-to-denials ratio against parity"});

  svg.appendChild(txt(CX0,20,"submissions",{fill:"var(--text-muted)","font-size":11}));
  svg.appendChild(txt(RX0,20,"claims / denials",{fill:"var(--text-muted)","font-size":11}));

  // count gridlines
  for(let v=0;v<=cmax;v+=500){
    svg.appendChild(el("line",{x1:cx(v),y1:TOP-8,x2:cx(v),y2:TOP+RUNS.length*ROW-6,
      stroke:"var(--grid)","stroke-width":1}));
    svg.appendChild(txt(cx(v),TOP+RUNS.length*ROW+14,fmt(v),
      {fill:"var(--text-muted)","font-size":10.5,"text-anchor":"middle"}));
  }
  // ratio axis + parity reference
  [0,0.5,1.0].forEach(v=>{
    svg.appendChild(el("line",{x1:rx(v),y1:TOP-8,x2:rx(v),y2:TOP+RUNS.length*ROW-6,
      stroke:v===1?"var(--axis)":"var(--grid)","stroke-width":1}));
    svg.appendChild(txt(rx(v),TOP+RUNS.length*ROW+14,v.toFixed(1),
      {fill:"var(--text-muted)","font-size":10.5,"text-anchor":"middle"}));
  });
  svg.appendChild(txt(rx(1.0),TOP-14,"parity",{fill:"var(--text-secondary)","font-size":10.5,"text-anchor":"middle"}));

  RUNS.forEach((r,i)=>{
    const y=TOP+i*ROW;
    svg.appendChild(txt(0,y+18,r.label,{fill:"var(--text-primary)","font-size":12.5,
      "font-weight":r.kind==="base"?600:400}));
    svg.appendChild(txt(0,y+33,r.note,{fill:"var(--text-muted)","font-size":10.5}));

    const bc=bar(CX0,y+4,cx(r.claims)-CX0,18,"var(--claims)");
    hover(bc,`<b>${r.label}</b><br>claims ${fmt(r.claims)}`); svg.appendChild(bc);
    const bd=bar(CX0,y+24,cx(r.denials)-CX0,18,"var(--denials)");   // 2px surface gap
    hover(bd,`<b>${r.label}</b><br>denials ${fmt(r.denials)}`); svg.appendChild(bd);

    svg.appendChild(txt(cx(r.claims)+6,y+17,fmt(r.claims),{fill:"var(--text-secondary)","font-size":10.5}));
    svg.appendChild(txt(cx(r.denials)+6,y+37,fmt(r.denials),{fill:"var(--text-secondary)","font-size":10.5}));

    const dot=el("circle",{cx:rx(r.ratio),cy:y+21,r:6,fill:"var(--claims)",
      stroke:"var(--surface-1)","stroke-width":2});
    hover(dot,`<b>${r.label}</b><br>ratio ${r.ratio.toFixed(3)} — ${r.note}`); svg.appendChild(dot);
    svg.appendChild(txt(rx(r.ratio),y+39,r.ratio.toFixed(3),
      {fill:"var(--text-secondary)","font-size":10.5,"text-anchor":"middle"}));
  });
  document.getElementById("f1").appendChild(svg);

  document.getElementById("t1").innerHTML=
    "<table><tr><th>scenario</th><th>claims</th><th>denials</th><th>ratio</th><th>basis</th></tr>"+
    RUNS.map(r=>`<tr><td>${r.label}</td><td>${fmt(r.claims)}</td><td>${fmt(r.denials)}</td>`+
      `<td>${r.ratio.toFixed(3)}</td><td>${r.kind==="estimate"?"pattern estimate":"recomputation"}</td></tr>`).join("")+
    "</table>";
})();

// ---------------------------------------------------- figure 2 · causes
(function(){
  const W=900, ROW=54, TOP=40, LAB=200, ZX=470, SPAN=330,
        H=TOP+CAUSES.length*ROW+38;
  const dmax=0.07;
  const dx=v=>ZX+(v/dmax)*SPAN*0.62;
  const svg=el("svg",{width:"100%",viewBox:`0 0 ${W} ${H}`,role:"img",
    "aria-label":"Change in the claims-to-denials ratio caused by each defect"});

  svg.appendChild(txt(ZX,20,"change in ratio",{fill:"var(--text-muted)","font-size":11,"text-anchor":"middle"}));
  [-0.02,0,0.02,0.04,0.06].forEach(v=>{
    svg.appendChild(el("line",{x1:dx(v),y1:TOP-8,x2:dx(v),y2:TOP+CAUSES.length*ROW-8,
      stroke:v===0?"var(--axis)":"var(--grid)","stroke-width":1}));
    svg.appendChild(txt(dx(v),TOP+CAUSES.length*ROW+12,(v>0?"+":"")+v.toFixed(2),
      {fill:"var(--text-muted)","font-size":10.5,"text-anchor":"middle"}));
  });

  CAUSES.forEach((c,i)=>{
    const y=TOP+i*ROW;
    svg.appendChild(txt(0,y+15,`${c.id} · ${c.label}`,{fill:"var(--text-primary)","font-size":12.5}));
    const words=c.detail.split(" "); let line="",ln=0;
    words.forEach(w=>{ if((line+w).length>62){ svg.appendChild(txt(0,y+29+ln*12,line,
        {fill:"var(--text-muted)","font-size":10})); line=w+" "; ln++; } else line+=w+" "; });
    svg.appendChild(txt(0,y+29+ln*12,line,{fill:"var(--text-muted)","font-size":10}));

    if(Math.abs(c.delta)<0.0005){
      svg.appendChild(el("line",{x1:ZX-4,y1:y+6,x2:ZX+4,y2:y+6,stroke:"var(--axis)","stroke-width":2}));
      svg.appendChild(txt(ZX+12,y+10,"no effect measured",{fill:"var(--text-muted)","font-size":10.5}));
    } else {
      const pos=c.delta>0, w=Math.abs(dx(c.delta)-ZX), col=pos?"var(--up)":"var(--down)";
      const b=pos?bar(ZX,y-2,w,17,col):barL(ZX,y-2,w,17,col);
      hover(b,`<b>${c.id} · ${c.label}</b><br>ratio ${(c.delta>0?"+":"")}${c.delta.toFixed(3)}`+
        `<br>${c.measured?"recomputation":"pattern estimate — over-counts"}`);
      svg.appendChild(b);
      svg.appendChild(txt(pos?dx(c.delta)+7:dx(c.delta)-7,y+11,
        (pos?"+":"")+c.delta.toFixed(3)+(c.measured?"":"  (upper bound)"),
        {fill:"var(--text-secondary)","font-size":10.5,"text-anchor":pos?"start":"end"}));
    }
  });
  document.getElementById("f2").appendChild(svg);

  document.getElementById("t2").innerHTML=
    "<table><tr><th>defect</th><th>change in ratio</th><th>basis</th></tr>"+
    CAUSES.map(c=>`<tr><td>${c.id} · ${c.label}</td><td>${(c.delta>0?"+":"")}${c.delta.toFixed(3)}</td>`+
      `<td>${c.measured?"recomputation":"pattern estimate (over-counts)"}</td></tr>`).join("")+
    "</table>";
})();
</script>
</div>
"""

display(HTML(FIGURES))


## What this notebook establishes, and what it does not

**Establishes**

- The published direction survives every measured defect. Denials outnumber claims in all five
  scenarios; the largest single effect moves the ratio from 0.805 to 0.865.
- The dropped rows are not a random sample. Among the 354 with prose comments, the ratio is 111
  claims to 23 denials — roughly six times more claim-rich than the corpus. That is why restoring
  them moves the ratio, and it belongs in any correction note.
- The sentence splitter defect, real in principle, changes **0** of 32,562 submissions here.
- Sentence-level citation attribution reproduces the published classifier's flags on all 4,012
  citing submissions.
- Two variants carry the pattern the study is about, confirmed by reading, out of four flagged.
  Both are the same submitter pair — Dasa asserting, LabCorp (Women's Health and Genetics)
  denying — so they are one pair observed twice, not two independent observations.

**Does not establish**

- The tier-1 estimate in section 6 over-counts. Reading its printed examples, most are genuine
  claims that happen to carry a caveat such as "further studies are needed". The honest figure is
  smaller than 17 and requires reading `C_candidate_false_claims.csv`.
- Two classifier weaknesses remain live and are qualitative, not quantified here. An inserted
  adjective can break a denial pattern — *"no variant specific experimental evidence ... has been
  reported"* is read as a claim. And a submission stating that an assay exists but is not an approved
  assay is a category the classifier has no label for.
- Counts of citations are counts of **PMIDs in comment text**. References given by author and year,
  and references in ClinVar's structured citation field, are invisible to this pipeline.
- Nothing here measures whether any laboratory's reading of the literature is correct.
